# Assignment-2 Query Expansion

 Set up 

In [1]:
!pip show chromadb

Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: 
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: C:\Users\Palak\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 


In [2]:
import sys
print(sys.executable)

c:\Users\Palak\Desktop\Assignments\assignment-2\.venv\Scripts\python.exe


In [3]:
import chromadb

from langchain_chroma import Chroma




In [4]:
import os
from dotenv import load_dotenv
GROQ_API_KEY=os.getenv('GROQ_API_KEY')

In [5]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

In [6]:
model_name = 'openai/gpt-oss-120b'

 Instantiating the embedding model

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [9]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime
# from google.colab import userdata

 Database creation

In [11]:
pdf_folder_location = "tesla-annual-reports"


In [12]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [13]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [ ]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
len(tesla_10k_chunks)

In [17]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [16]:
!pip show sentence-transformers

Name: sentence-transformers
Version: 5.5.1
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: Apache 2.0
Location: C:\Users\palak\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages
Requires: huggingface-hub, numpy, scikit-learn, scipy, torch, tqdm, transformers, typing_extensions
Required-by: 


In [18]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [19]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [20]:
chromadb_client.heartbeat()

1780489681285914700

In [21]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [22]:
chromadb_client.count_collections()

1

In [32]:
i = 0 # Initialize the starting index for the chunks

while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
    vectorstore.add_documents( # Add documents to the vector store in batches of 500
        documents=tesla_10k_chunks[i:i+500], # Get the current batch of 500 chunks
        ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
    )

    i += 500 # Increment the index by 500 to move to the next batch
    time.sleep(5) # Pause for 30 seconds to avoid rate limiting issues with the vector store

In [23]:
chromadb_client

In [23]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [25]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002674756AA90>, search_kwargs={'k': 5})

Query Expansion

In [24]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.
Generate variants from at least four angles: financial analyst phrasing, risk-factor phrasing, operational phrasing, and synonym/subtopic phrasing.
Expansion must preserve the original user intent. Do not create unrelated queries. 

Return at least 4 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [40]:
client = GroqClient()

Query expantion for all 4 questions simultaneously

In [45]:
from groq_client import GroqClient
import json

# Initialize client
client = GroqClient()

# List of user queries
user_inputs = [
    "Does Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A.",

    "Explain how Tesla's AI and product roadmap is reflected in spending, operational priorities, and risk disclosures. Avoid generic AI claims.",

    "A supplier asks whether Tesla is exposed to concentration risk across factories, suppliers, raw materials, or geographies. Prepare a concise risk note with citations.",

    "Compare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation?"
]

# Store all results
results = []

# Generate expanded questions for each query
for idx, user_query in enumerate(user_inputs, start=1):
    questions = client.generate(
        system_prompt=query_expansion_system_message,
        user_prompt=user_message_template.format(question=user_query)
    )

    # Convert multiline output into a list of questions
    expanded_queries = [
        q.strip()
        for q in questions.split("\n")
        if q.strip()
    ]

    # Create JSON object
    result = {
        "question_id": f"Q{idx}",
        "original_query": user_query,
        "expanded_queries": expanded_queries
    }

    results.append(result)

# Print JSON
print(json.dumps(results, indent=2))

# Optional: Save to file
with open("query_expansions.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to query_expansions.json")

[
  {
    "question_id": "Q1",
    "original_query": "Does Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A.",
    "expanded_queries": [
      "From a financial analyst perspective, is Tesla's growth more limited by external supply chain risks or by its internal execution and cost structure, based on the Risk Factors and MD&A sections?",
      "Do the Risk Factors and MD&A indicate that Tesla's growth trajectory is primarily constrained by external supply risks rather than internal operational execution and cost management?",
      "Considering Tesla's operational performance, does the evidence in the Risk Factors and MD&A suggest that external supply constraints or internal execution and cost structure are the main barriers to growth?",
      "Is Tesla's ability to expand hindered more by outside supply chain uncertainties or by its own execution efficiency and cost framework, accor

In [ ]:
# response = client.chat.completions.create(
#     model=model_name,
#     messages=[
#         {"role": "system", "content": query_expansion_system_message},
#         {"role": "user", "content": user_message_template.format(question=user_input)}
#     ],
#     temperature=0
# )

Context Retreival for all 4 queries

In [51]:
import json
import re

def clean_text(text):
    text = text.replace("\t", " ").replace("\n", " ").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

final_results = []

for item in results:
    expanded_context_list = []

    for que in item["expanded_queries"]:
        docs = retriever.invoke(que)

        expanded_context_list.extend(
            [clean_text(doc.page_content) for doc in docs]
        )

    # Remove duplicate contexts
    final_context_documents = list(set(expanded_context_list))

    final_results.append({
        "question_id": item["question_id"],
        "original_query": item["original_query"],
        "expanded_queries": item["expanded_queries"],
        "retrieved_contexts": final_context_documents
    })

# Save to JSON file
output_file = "retrieval_results.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        final_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Saved {len(final_results)} queries to {output_file}")

Saved 4 queries to retrieval_results.json


LLM Call to generate output using retrieved context

In [52]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".

"""

In [53]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [54]:
import json
from groq_client import GroqClient

client = GroqClient()

# Load retrieval results
with open("retrieval_results.json", "r", encoding="utf-8") as f:
    retrieval_results = json.load(f)

# Generate answer for each query
for item in retrieval_results:

    context = "\n\n".join(item["retrieved_contexts"])

    answer = client.generate(
        system_prompt=qna_system_message,
        user_prompt=qna_user_message_template.format(
            context=context,
            question=item["original_query"]
        )
    )

    # Add answer to JSON object
    item["answer"] = answer

    # Print on screen
    print("=" * 100)
    print(f"Question ID : {item['question_id']}")
    print(f"Question     : {item['original_query']}")
    print(f"Answer       : {answer}")
    print("=" * 100)

# Save updated JSON
with open("retrieval_results.json", "w", encoding="utf-8") as f:
    json.dump(
        retrieval_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nUpdated retrieval_results.json with generated answers.")

Question ID : Q1
Question     : Does Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A.
Answer       : Tesla’s growth narrative is portrayed as being limited more by internal execution and cost‑structure challenges than by external supply constraints.  

- The Risk Factors repeatedly stress the company’s need to “rapidly increase the number of our Supercharger stations and connectors,” to “avoid cost overruns,” and to “hire additional personnel” to support expansion.  It warns that failure to “manage our growth effectively” could harm the brand, financial condition and operating results.  
- The MD&A notes that Tesla must “scale manufacturing, delivery and service operations to meet demand,” and that “delays in scaling… could affect quarterly production and sales performance.”  It also highlights “inflation caused by the COVID‑19 pandemic and general global economic conditions” and th

 Original Query → Expanded Queries → Retrieved Contexts → Final Answer